Ejercicio 1.
Crea un DataFrame de PySpark a partir de un archivo csv cualquiera usando el método spark.read.csv() .
Muestra el DataFrame resultante.


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, lit, array, size, create_map, struct, count
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

df_csv = spark.read.csv("/Workspace/sampleFiles/data_science_salaries(in).csv", header=True, inferSchema=True)
print("\n--- Resultado ---")
df_csv.show()


--- Resultado ---
+--------------------+----------------+---------------+-----------+---------+------------------+------+---------------+-------------+----------------+------------+
|           job_title|experience_level|employment_type|work_models|work_year|employee_residence|salary|salary_currency|salary_in_usd|company_location|company_size|
+--------------------+----------------+---------------+-----------+---------+------------------+------+---------------+-------------+----------------+------------+
|       Data Engineer|       Mid-level|      Full-time|     Remote|     2024|     United States|148100|            USD|       148100|   United States|      Medium|
|       Data Engineer|       Mid-level|      Full-time|     Remote|     2024|     United States| 98700|            USD|        98700|   United States|      Medium|
|      Data Scientist|    Senior-level|      Full-time|     Remote|     2024|     United States|140032|            USD|       140032|   United States|      Mediu

Ejercicio 2

Tienes un archivo de salarios de Data Scientists de empresas de distintos tamaños, desde pequeñas
hasta grandes. Quieres ver si existe una diferencia importante entre los salarios promedio agrupados por
tamaño de la empresa.

✅ Instrucciones

Carga un archivo CSV como un DataFrame e infiere el esquema.

Devuelve el conteo del número de filas.

Agrupa por la columna company_size y calcula el salario promedio con salary_in_usd .

In [0]:
df_csv = spark.read.csv("/Workspace/sampleFiles/data_science_salaries(in).csv", header=True, inferSchema=True)
df_csv.groupby("company_size").count().show()
salario_promedio = df_csv.groupby("company_size").avg("salary_in_usd")
display(salario_promedio)

+------------+-----+
|company_size|count|
+------------+-----+
|      Medium| 5860|
|       Small|  170|
|       Large|  569|
+------------+-----+



company_size,avg(salary_in_usd)
Medium,149659.3866894198
Small,87687.4588235294
Large,120638.40421792619


Ejercicio 3.
Usando el mismo conjunto de datos del ejercicio anterior, te diste cuenta de que solo te interesa conocer
los trabajos que son de nivel inicial ( "Entry-level" ) en Canadá. ¿Cómo se ven los salarios allí?
✅ Instrucciones

Filtra para crear un subconjunto del DataFrame donde company_location sea "CANADA" .

Calcula el promedio de la columna salary_in_usd .

¡Muestra el resultado!


In [0]:
from pyspark.sql.functions import col

df_canada = df_csv.filter((col("company_location") == "Canada") & (col("experience_level") == "Entry-level"))

print("--- Trabajos Entry-Level en Canadá ---")
df_canada.show()

--- Trabajos Entry-Level en Canadá ---
+--------------------+----------------+---------------+-----------+---------+------------------+------+---------------+-------------+----------------+------------+
|           job_title|experience_level|employment_type|work_models|work_year|employee_residence|salary|salary_currency|salary_in_usd|company_location|company_size|
+--------------------+----------------+---------------+-----------+---------+------------------+------+---------------+-------------+----------------+------------+
|Business Intellig...|     Entry-level|      Full-time|    On-site|     2023|            Canada| 91875|            USD|        91875|          Canada|      Medium|
|Business Intellig...|     Entry-level|      Full-time|    On-site|     2023|            Canada| 55125|            USD|        55125|          Canada|      Medium|
|       Data Engineer|     Entry-level|      Full-time|     Hybrid|     2023|     United States| 90000|            USD|        90000|        

Ejercicio 4.Buscar tres ficheros: 

    Uno en csv, 

    otro en .json y 
    
    otro en.parquet 
que hagan todos referencia al mismo esquema. 

Con PySpark, crear el dataframe para cada archivo, luego unir los tres
dataframes en un único dataframe final. Finalmente exportar tu
dataframe final a un archivo .csv.


In [0]:
df_csv = spark.read.csv("/Workspace/sampleFiles/data_science_salaries(in).csv", inferSchema=True, header=True)
df_json = spark.read.option("inferSchema", "true").json("/Workspace/sampleFiles/data_science_salaries.json", multiLine=True)
df_parquet = spark.read.parquet("/Workspace/sampleFiles/data_science_salaries.parquet")

df_csv.printSchema()
df_json.printSchema()
df_parquet.printSchema()

df_final = df_csv.unionByName(df_json).unionByName(df_parquet)
df_final.coalesce(1).write.mode("overwrite").option("header", "true").csv("/Volumes/workspace/default/file")

root
 |-- job_title: string (nullable = true)
 |-- experience_level: string (nullable = true)
 |-- employment_type: string (nullable = true)
 |-- work_models: string (nullable = true)
 |-- work_year: integer (nullable = true)
 |-- employee_residence: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- salary_currency: string (nullable = true)
 |-- salary_in_usd: integer (nullable = true)
 |-- company_location: string (nullable = true)
 |-- company_size: string (nullable = true)

root
 |-- company_location: string (nullable = true)
 |-- company_size: string (nullable = true)
 |-- employee_residence: string (nullable = true)
 |-- employment_type: string (nullable = true)
 |-- experience_level: string (nullable = true)
 |-- job_title: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- salary_currency: string (nullable = true)
 |-- salary_in_usd: long (nullable = true)
 |-- work_models: string (nullable = true)
 |-- work_year: long (nullable = true)

root
 |

Ejercicio 5.

Imagina que tienes un conjunto de datos de un censo que sabes que tiene un encabezado y un esquema. Vamos a cargar ese conjunto de datos y dejar que PySpark infiera el esquema.

¿Qué ves si filtras adultos mayores de 40 años?

✅ Instrucciones

Carga un archivo JSON censo.json .

Filtra los datos para incluir personas con age mayor que 40 .

Muestra los resultados.


In [0]:
df_json_censo = spark.read.option("inferSchema", "true").json("/Workspace/sampleFiles/censo.json", multiLine=True)
edad_mayor_40 = df_json_censo.filter(col("age") > 40).orderBy(col("age").desc())
edad_mayor_40.show()

+---+-------------+------+--------------+-------------+
|age|education_num|income|marital_status|   occupation|
+---+-------------+------+--------------+-------------+
| 69|            8| <=50K|        Single|        Sales|
| 69|           16|  >50K|       Married| Craft-repair|
| 68|            6|  >50K|       Married| Craft-repair|
| 66|           16| <=50K|        Single|Other-service|
| 66|            2| <=50K|        Single|        Sales|
| 63|            7|  >50K|       Widowed|Other-service|
| 62|            8|  >50K|       Married| Tech-support|
| 61|           11|  >50K|       Widowed| Craft-repair|
| 60|           13|  >50K|      Divorced|        Sales|
| 59|            9|  >50K|       Widowed| Craft-repair|
| 56|            9| <=50K|       Married| Tech-support|
| 50|            5| <=50K|      Divorced|        Sales|
| 50|            4|  >50K|      Divorced|        Sales|
| 46|            3| <=50K|       Married|Other-service|
| 45|           15|  >50K|       Married|       

Ejercicio 6
Vas a definir un esquema directamente. Para ello usaras el siguiente diccionario de datos:
    
Variable Descripción

age Edad individual

education_num Educación por grado

marital_status Estado civil

occupation Ocupación

income Ingreso categórico


✅ Instrucciones

Especifica el esquema de datos, asignando nombres a las columnas ( age , education_num , marital_status ,
occupation , e income ) y los tipos de columna, estableciendo una coma para el argumento sep= .
Lee los datos de un archivo delimitado por comas llamado adult.csv .
Imprime el esquema del DataFrame resultante.


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
schema_adult = StructType([
    StructField("age", IntegerType(), True),
    StructField("education_num", IntegerType(), True),
    StructField("marital_status", StringType(), True),
    StructField("occupation", StringType(), True),
    StructField("income", StringType(), True)
])
df_json_adult = spark.read.csv("/Workspace/sampleFiles/adult(in).csv", schema=schema_adult, sep=",", header=False)
print("--- Esquema del DataFrame 'adult' ---")
df_json_adult.printSchema()
# (Opcional) Ver los datos
df_json_adult.show(5)

--- Esquema del DataFrame 'adult' ---
root
 |-- age: integer (nullable = true)
 |-- education_num: integer (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- income: string (nullable = true)

+---+-------------+--------------+---------------+------+
|age|education_num|marital_status|     occupation|income|
+---+-------------+--------------+---------------+------+
| 66|           14|        Single|   Tech-support|  >50K|
| 44|           11|      Divorced|          Sales| <=50K|
| 25|            1|       Widowed|Exec-managerial| <=50K|
| 34|            8|        Single|          Sales| <=50K|
| 38|            7|       Widowed|   Tech-support|  >50K|
+---+-------------+--------------+---------------+------+
only showing top 5 rows


Ejercicio 7
Tienes un fichero que se llama income.csv que tiene valores faltantes y valores nulos. Hay que limpiarlo.
Elimina las filas que tengan algún valor nulo o faltante y muestra los resultados.

✅ Instrucciones
Con .show() imprime todos los registros del dataset (tiene 40)

Elimina cualquier fila con valores nulos y los datos faltantes cámbialos por 1 en el DataFrame
income.csv.

Elimina todos los registros que tengan valor = 1

Muestra el dataframe resultante

Cambia el nombre de la columna income por salario

Crear una nueva columna con nombre weekly_salary dividiendo la columna salario entre 52.

Muestra el dataframe resultante.


In [0]:
df_income = spark.read.csv("/Workspace/sampleFiles/income(in).csv", inferSchema=True, header=True)
df_noNull = df_income.na.drop() # sin NULLS
df_clean_and_fill = df_income.na.fill(1).na.fill("1") # Nulls numeric y strings a 1
df_sin_unos = df_clean_and_fill.filter(col("education_num") != "1") # filtro para quedarse con filas distintas a 1
df_income_1 = df_clean_and_fill.withColumnRenamed("income", "salario") # Cambio de nombre
df_weekly_salary = df_income_1.withColumn("weekly_salary", df_income_1["salario"] / 52) #Añadir columna
print("---NORMAL---")
df_income.display()
print("---SIN NULOS---")
df_noNull.display()
print("---NULOS A 1---")
df_clean_and_fill.display()
print("---FILAS QUE NO TIENEN 1---")
df_sin_unos.display()
print("---CAMBIO DE NOMBRE COL---")
df_income_1.display()
print("---NUEVA COL---")
df_weekly_salary.display()

---NORMAL---


age,education_num,marital_status,occupation,income
56.0,7.0,Single,Craft-repair,106753.0
30.0,1.0,Widowed,Craft-repair,90614.0
26.0,8.0,Widowed,Exec-managerial,61308.0
56.0,8.0,Widowed,Other-service,119890.0
27.0,15.0,Single,Exec-managerial,118106.0
68.0,5.0,Married,Sales,81776.0
31.0,4.0,Single,Tech-support,62739.0
55.0,6.0,Married,Tech-support,74602.0
44.0,14.0,Divorced,Tech-support,79081.0
40.0,8.0,Divorced,Exec-managerial,87994.0


---SIN NULOS---


age,education_num,marital_status,occupation,income
56.0,7.0,Single,Craft-repair,106753.0
30.0,1.0,Widowed,Craft-repair,90614.0
26.0,8.0,Widowed,Exec-managerial,61308.0
56.0,8.0,Widowed,Other-service,119890.0
27.0,15.0,Single,Exec-managerial,118106.0
68.0,5.0,Married,Sales,81776.0
31.0,4.0,Single,Tech-support,62739.0
55.0,6.0,Married,Tech-support,74602.0
44.0,14.0,Divorced,Tech-support,79081.0
40.0,8.0,Divorced,Exec-managerial,87994.0


---NULOS A 1---


...

age,education_num,marital_status,occupation,income
56.0,7.0,Single,Craft-repair,106753.0
26.0,8.0,Widowed,Exec-managerial,61308.0
56.0,8.0,Widowed,Other-service,119890.0
27.0,15.0,Single,Exec-managerial,118106.0
68.0,5.0,Married,Sales,81776.0
31.0,4.0,Single,Tech-support,62739.0
55.0,6.0,Married,Tech-support,74602.0
44.0,14.0,Divorced,Tech-support,79081.0
40.0,8.0,Divorced,Exec-managerial,87994.0
69.0,4.0,Widowed,Sales,36770.0


---CAMBIO DE NOMBRE COL---


age,education_num,marital_status,occupation,salario
56.0,7.0,Single,Craft-repair,106753.0
30.0,1.0,Widowed,Craft-repair,90614.0
26.0,8.0,Widowed,Exec-managerial,61308.0
56.0,8.0,Widowed,Other-service,119890.0
27.0,15.0,Single,Exec-managerial,118106.0
68.0,5.0,Married,Sales,81776.0
31.0,4.0,Single,Tech-support,62739.0
55.0,6.0,Married,Tech-support,74602.0
44.0,14.0,Divorced,Tech-support,79081.0
40.0,8.0,Divorced,Exec-managerial,87994.0


---NUEVA COL---


age,education_num,marital_status,occupation,salario,weekly_salary
56.0,7.0,Single,Craft-repair,106753.0,2052.9423076923076
30.0,1.0,Widowed,Craft-repair,90614.0,1742.576923076923
26.0,8.0,Widowed,Exec-managerial,61308.0,1179.0
56.0,8.0,Widowed,Other-service,119890.0,2305.576923076923
27.0,15.0,Single,Exec-managerial,118106.0,2271.269230769231
68.0,5.0,Married,Sales,81776.0,1572.6153846153845
31.0,4.0,Single,Tech-support,62739.0,1206.5192307692307
55.0,6.0,Married,Tech-support,74602.0,1434.6538461538462
44.0,14.0,Divorced,Tech-support,79081.0,1520.7884615384614
40.0,8.0,Divorced,Exec-managerial,87994.0,1692.1923076923076


# Ejercicios clase 38

## Ejercicio 1:

Has sido contratado para una compañía global de viajes. Tu primera tarea es ayudar a la empresa a mejorar sus operaciones analizando los datos de vuelos. 

Tienes **dos dataset** en tu espacio de trabajo:

- Uno que contiene detalles sobre los vuelos ([`flights`](https://tajamar365.sharepoint.com/:x:/s/3431-MasterIA2025-2026/EVx8S-FhnDZLiCvqv0NXb2kBd7JLXcybP4_pyrknGIbhRg?e=PreVUB)).
- Otro que tiene información sobre los aeropuertos de destino ([`airports`](https://tajamar365.sharepoint.com/:x:/s/3431-MasterIA2025-2026/EUS5AAmLuSBAi6hOEpHYjX0BqZs3g9kYPNXu9J3cpBR5bw?e=OUA33L)).

Combina estos conjuntos de datos para crear un dataset que relacione cada vuelo con su **aeropuerto de destino**.

### ✅ **Instrucciones**

- Examina el DataFrame `airports`. Observa qué columna clave te permitirá unir `airports` con la tabla `flights`.
- Une el DataFrame `flights` con `airports` usando la columna `"dest"`.
    
    Guarda el resultado como `flights_with_airports`.
    
- Examina de nuevo `flights_with_airports`. Observa la nueva información que se ha añadido. Muestra el resultado
</aside>

In [0]:
df_airports = spark.read.csv("/Workspace/Users/inigosamuel.jimenez@tajamar365.com/airports.csv", header=True, inferSchema=True)
df_flights = spark.read.csv("/Workspace/Users/inigosamuel.jimenez@tajamar365.com/flights.csv", header=True, inferSchema=True)

df_airports.describe()
display(df_airports)

df_flights.describe()
display(df_flights)

airport_code,airport_name,city,state
LAX,Los Angeles International,Los Angeles,CA
SFO,San Francisco International,San Francisco,CA
MIA,Miami International,Miami,FL
SEA,Seattle-Tacoma International,Seattle,WA
JFK,John F. Kennedy International,New York,NY


flight_id,airline,origin,dest,departure_time
1,Delta,JFK,LAX,2025-11-26T08:00:00.000Z
2,United,ORD,SFO,2025-11-26T09:30:00.000Z
3,American,ATL,MIA,2025-11-26T10:15:00.000Z
4,Southwest,LAX,SEA,2025-11-26T12:00:00.000Z
5,Alaska,SEA,JFK,2025-11-26T14:45:00.000Z


In [0]:
flights_with_airports = df_airports.join(df_flights, df_airports["airport_code"] == df_flights["dest"], how="inner")
display(flights_with_airports)

airport_code,airport_name,city,state,flight_id,airline,origin,dest,departure_time
LAX,Los Angeles International,Los Angeles,CA,1,Delta,JFK,LAX,2025-11-26T08:00:00.000Z
SFO,San Francisco International,San Francisco,CA,2,United,ORD,SFO,2025-11-26T09:30:00.000Z
MIA,Miami International,Miami,FL,3,American,ATL,MIA,2025-11-26T10:15:00.000Z
SEA,Seattle-Tacoma International,Seattle,WA,4,Southwest,LAX,SEA,2025-11-26T12:00:00.000Z
JFK,John F. Kennedy International,New York,NY,5,Alaska,SEA,JFK,2025-11-26T14:45:00.000Z


## Ejercicio 2:

Has sido contratado para una compañía. Tienes dos archivos CSV:

- Uno con información de empleados del **Departamento A: [dept_a.csv](https://tajamar365.sharepoint.com/:x:/s/3431-MasterIA2025-2026/ETLSCKiVZVROnI8EfEOJn4AB6sKcoYLha2ZnfxXIuouG4w?e=mvoZPd)**
- Otro con empleados del **Departamento B: [dept_b.csv](https://tajamar365.sharepoint.com/:x:/s/3431-MasterIA2025-2026/EVuezLyY7A1Jpyb0WdyUyGEBaXTdclXirMT3wIxgVNFDtg?e=IJVjWh)**

Tu tarea es combinarlos en un solo DataFrame para analizar todos los empleados juntos.

### 🎯 **Instrucciones**

1️⃣ Carga ambos archivos como DataFrames en PySpark.

2️⃣ Verifica que ambos DataFrames tengan **el mismo esquema**.

3️⃣ Aplica `.union()` para unir las filas de ambos DataFrames en uno solo llamado `all_employees`.

4️⃣ Muestra el resultado final.

</aside>

In [0]:
dept_a = spark.read.csv("/Workspace/Users/inigosamuel.jimenez@tajamar365.com/dept_a.csv", header=True, inferSchema=True)
dept_b = spark.read.csv("/Workspace/Users/inigosamuel.jimenez@tajamar365.com/dept_b.csv", header=True, inferSchema=True)

dept_a.printSchema()
dept_b.printSchema()

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)



In [0]:
all_employees = dept_a.union(dept_b)
display(all_employees)

emp_id,name,department,salary
1,Alice,A,50000
2,Bob,A,55000
3,Charlie,A,60000
4,David,B,52000
5,Eva,B,58000
6,Frank,B,61000


## Ejercicio 3:

Combina varias columnas de calificaciones en un solo array usando PySpark y practica operaciones sobre ese array.

### ✅ Instrucciones

1️⃣ Carga el archivo [`students.csv`](https://tajamar365.sharepoint.com/:x:/s/3431-MasterIA2025-2026/EQJfhcRXl7pJt1Jhhdun_gIB4w5BmQgFmzxpiHFlQ6fAfQ?e=ghBifI) como DataFrame.

2️⃣ Usa la función `array()` para combinar las columnas `score1`, `score2` y `score3` en una nueva columna llamada `scores_array`.

3️⃣ Muestra el DataFrame resultante para ver la nueva columna `scores_array`.

4️⃣ Calcula la longitud del array de calificaciones para cada estudiante usando `size()`.

</aside>

In [0]:
students = spark.read.csv("/Workspace/Users/inigosamuel.jimenez@tajamar365.com/students.csv", header=True, inferSchema=True)
display(students)

student_id,name,score1,score2,score3
1,Juan,71,92,89
2,Ana,99,95,97
3,Luis,91,81,76
4,Marta,82,75,89
5,Carlos,95,95,70
6,Elena,79,79,79
7,Pedro,79,83,74
8,Lucía,90,82,91
9,Miguel,83,98,80
10,Sara,97,80,82


In [0]:
from pyspark.sql.functions import array, col, size

students = students.withColumn("scores_array", array(col("score1"), col("score2"), col("score3")))
display(students)

student_id,name,score1,score2,score3,scores_array
1,Juan,71,92,89,"List(71, 92, 89)"
2,Ana,99,95,97,"List(99, 95, 97)"
3,Luis,91,81,76,"List(91, 81, 76)"
4,Marta,82,75,89,"List(82, 75, 89)"
5,Carlos,95,95,70,"List(95, 95, 70)"
6,Elena,79,79,79,"List(79, 79, 79)"
7,Pedro,79,83,74,"List(79, 83, 74)"
8,Lucía,90,82,91,"List(90, 82, 91)"
9,Miguel,83,98,80,"List(83, 98, 80)"
10,Sara,97,80,82,"List(97, 80, 82)"


In [0]:
students.select(
    col("scores_array"),
    size(col("scores_array")).alias("size_of_scores")
).show(5)

+------------+--------------+
|scores_array|size_of_scores|
+------------+--------------+
|[71, 92, 89]|             3|
|[99, 95, 97]|             3|
|[91, 81, 76]|             3|
|[82, 75, 89]|             3|
|[95, 95, 70]|             3|
+------------+--------------+
only showing top 5 rows


## Ejercicio 4:

Combina varias columnas de atributos de producto en un solo `map` de clave-valor usando PySpark y explora cómo acceder a sus pares.

### ✅ Instrucciones

1️⃣ Carga el archivo [`products.csv`](https://tajamar365.sharepoint.com/:x:/s/3431-MasterIA2025-2026/ESOQE5FM2-5IiArG9ds-LL4Bd3v-2jPzmCOMWEIl0afKSA?e=ccxYUZ) como DataFrame.

2️⃣ Usa la función `create_map()` para combinar las columnas `brand`, `warranty_years` y `color` en una nueva columna llamada `properties_map`.

3️⃣ Muestra el DataFrame resultante para ver la nueva columna `properties_map`.

4️⃣ Accede a un valor específico del map (por ejemplo, la clave `"brand"`).

</aside>

In [0]:
products = spark.read.csv("/Workspace/Users/inigosamuel.jimenez@tajamar365.com/products.csv", header=True, inferSchema=True)
display(products)

product_id,product_name,brand,warranty_years,color
1,Mouse,Dell,3,Blue
2,Monitor,Canon,3,Blue
3,Speaker,Apple,2,Silver
4,Headphones,HP,1,Blue
5,Mouse,LG,2,Silver
6,Camera,Fitbit,2,Black
7,Tablet,Apple,2,White
8,Mouse,Sony,2,White
9,Speaker,Apple,2,White
10,Headphones,HP,2,Blue


In [0]:
products.dtypes

[('product_id', 'int'),
 ('product_name', 'string'),
 ('brand', 'string'),
 ('warranty_years', 'int'),
 ('color', 'string'),
 ('properties_map', 'map<string,bigint>')]

In [0]:
from pyspark.sql.functions import create_map, col, lit
from pyspark.sql.types import StringType

products = products.withColumn(
    "properties_map",
    create_map(
        lit("brand"), col("brand").cast(StringType()),
        lit("warranty_years"), col("warranty_years").cast(StringType()),
        lit("color"), col("color").cast(StringType())
    )
)

display(products)

product_id,product_name,brand,warranty_years,color,properties_map
1,Mouse,Dell,3,Blue,"Map(brand -> Dell, color -> Blue, warranty_years -> 3)"
2,Monitor,Canon,3,Blue,"Map(brand -> Canon, color -> Blue, warranty_years -> 3)"
3,Speaker,Apple,2,Silver,"Map(brand -> Apple, color -> Silver, warranty_years -> 2)"
4,Headphones,HP,1,Blue,"Map(brand -> HP, color -> Blue, warranty_years -> 1)"
5,Mouse,LG,2,Silver,"Map(brand -> LG, color -> Silver, warranty_years -> 2)"
6,Camera,Fitbit,2,Black,"Map(brand -> Fitbit, color -> Black, warranty_years -> 2)"
7,Tablet,Apple,2,White,"Map(brand -> Apple, color -> White, warranty_years -> 2)"
8,Mouse,Sony,2,White,"Map(brand -> Sony, color -> White, warranty_years -> 2)"
9,Speaker,Apple,2,White,"Map(brand -> Apple, color -> White, warranty_years -> 2)"
10,Headphones,HP,2,Blue,"Map(brand -> HP, color -> Blue, warranty_years -> 2)"


In [0]:
products.select(
    col("properties_map"),
    col("properties_map")["brand"].alias("brand"),
).show()

+--------------------+-------+
|      properties_map|  brand|
+--------------------+-------+
|{brand -> Dell, w...|   Dell|
|{brand -> Canon, ...|  Canon|
|{brand -> Apple, ...|  Apple|
|{brand -> HP, war...|     HP|
|{brand -> LG, war...|     LG|
|{brand -> Fitbit,...| Fitbit|
|{brand -> Apple, ...|  Apple|
|{brand -> Sony, w...|   Sony|
|{brand -> Apple, ...|  Apple|
|{brand -> HP, war...|     HP|
|{brand -> Sony, w...|   Sony|
|{brand -> Fitbit,...| Fitbit|
|{brand -> HP, war...|     HP|
|{brand -> Samsung...|Samsung|
|{brand -> LG, war...|     LG|
|{brand -> Fitbit,...| Fitbit|
|{brand -> Samsung...|Samsung|
|{brand -> Canon, ...|  Canon|
|{brand -> Fitbit,...| Fitbit|
|{brand -> Canon, ...|  Canon|
+--------------------+-------+
only showing top 20 rows


## Ejercicio 5:

Combina varias columnas de dirección en un solo `Struct` usando PySpark y aprende a acceder a los campos internos de ese `Struct`.

### ✅ Instrucciones

1️⃣ Carga el archivo [`employees.csv`](https://tajamar365.sharepoint.com/:x:/s/3431-MasterIA2025-2026/EWzWpIP15h1MoptISglX48MBUIxRnJoDuguVAtc8huYh5Q?e=GTUX1p) como DataFrame.

2️⃣ Usa la función `struct()` para combinar las columnas `street`, `city` y `zip_code` en una nueva columna llamada `address_struct`.

3️⃣ Muestra el DataFrame resultante para ver la nueva columna `address_struct`.

4️⃣ Accede a un campo interno del `Struct` (por ejemplo, `city`).

</aside>

In [0]:
df_emp = spark.read.csv("/Workspace/Users/inigosamuel.jimenez@tajamar365.com/employees.csv", header=True, inferSchema=True)
display(df_emp)

employee_id,name,street,city,zip_code
1,Juan,Calle A,Madrid,28000
2,Ana,Calle B,Barcelona,28001
3,Luis,Calle C,Sevilla,28002
4,Marta,Calle D,Valencia,28003
5,Carlos,Calle E,Bilbao,28004
6,Elena,Calle F,Zaragoza,28005
7,Pedro,Calle G,Málaga,28006
8,Lucía,Calle H,Granada,28007
9,Miguel,Calle I,Murcia,28008
10,Sara,Calle J,Alicante,28009


In [0]:
from pyspark.sql.functions import struct
df_emp = df_emp.withColumn("adress_struct", struct("street", "city", "zip_code"))
display(df_emp)

employee_id,name,street,city,zip_code,adress_struct
1,Juan,Calle A,Madrid,28000,"List(Calle A, Madrid, 28000)"
2,Ana,Calle B,Barcelona,28001,"List(Calle B, Barcelona, 28001)"
3,Luis,Calle C,Sevilla,28002,"List(Calle C, Sevilla, 28002)"
4,Marta,Calle D,Valencia,28003,"List(Calle D, Valencia, 28003)"
5,Carlos,Calle E,Bilbao,28004,"List(Calle E, Bilbao, 28004)"
6,Elena,Calle F,Zaragoza,28005,"List(Calle F, Zaragoza, 28005)"
7,Pedro,Calle G,Málaga,28006,"List(Calle G, Málaga, 28006)"
8,Lucía,Calle H,Granada,28007,"List(Calle H, Granada, 28007)"
9,Miguel,Calle I,Murcia,28008,"List(Calle I, Murcia, 28008)"
10,Sara,Calle J,Alicante,28009,"List(Calle J, Alicante, 28009)"


In [0]:
df_emp.select(
    col("adress_struct"),
    col("adress_struct")["city"].alias("city")
).show()

+--------------------+----------+
|       adress_struct|      city|
+--------------------+----------+
|{Calle A, Madrid,...|    Madrid|
|{Calle B, Barcelo...| Barcelona|
|{Calle C, Sevilla...|   Sevilla|
|{Calle D, Valenci...|  Valencia|
|{Calle E, Bilbao,...|    Bilbao|
|{Calle F, Zaragoz...|  Zaragoza|
|{Calle G, Málaga,...|    Málaga|
|{Calle H, Granada...|   Granada|
|{Calle I, Murcia,...|    Murcia|
|{Calle J, Alicant...|  Alicante|
|{Calle K, Santand...| Santander|
|{Calle L, Toledo,...|    Toledo|
|{Calle M, Vallado...|Valladolid|
|{Calle N, Salaman...| Salamanca|
|{Calle O, Pamplon...|  Pamplona|
|{Calle P, León, 2...|      León|
|{Calle Q, Oviedo,...|    Oviedo|
|{Calle R, Logroño...|   Logroño|
|{Calle S, Burgos,...|    Burgos|
|{Calle T, Huelva,...|    Huelva|
+--------------------+----------+
only showing top 20 rows


## Ejercicio 6:

Combinar una lista de estudiantes con su información de becas usando un `LEFT JOIN`. Esto permite conservar **todos los estudiantes**, incluso los que **no tienen beca**.

### ✅ Instrucciones

1️⃣ Carga ambos archivos como DataFrames en PySpark.

2️⃣ Realiza un `LEFT JOIN` de  [`students_leftjoin.csv`](https://tajamar365.sharepoint.com/:x:/s/3431-MasterIA2025-2026/EYZx60jLPbdFvy6485bCXWsBB_781TjmZA4oZoiVeWAtkA?e=bdBlru) con [`scholarships_leftjoin.csv`](https://tajamar365.sharepoint.com/:x:/s/3431-MasterIA2025-2026/ERLdYI3hexhMjK4jmp3gA_sBPNdLoBIL147UQ1y6o7uQ4Q?e=yyij4N) usando la columna `student_id`.

3️⃣ Guarda el resultado como `students_with_scholarships`.

4️⃣ Muestra el resultado y verifica que **los estudiantes sin beca aparecen con valores `null` en `scholarship_type`**.

</aside>

In [0]:
df_stu_ljoin = spark.read.csv("/Workspace/Users/inigosamuel.jimenez@tajamar365.com/students_leftjoin.csv", header=True, inferSchema=True)
display(df_stu_ljoin)

df_schol_ljoin = spark.read.csv("/Workspace/Users/inigosamuel.jimenez@tajamar365.com/scholarships_leftjoin.csv", header=True, inferSchema=True)
display(df_schol_ljoin)

student_id,name,grade
1,Juan,85
2,Ana,90
3,Luis,78
4,Marta,92
5,Carlos,87
6,Elena,88
7,Pedro,75
8,Lucía,91
9,Miguel,80
10,Sara,89


student_id,scholarship_type
1,Full
2,Half
4,Full
5,Half
7,Quarter
9,Half
12,Full
14,Quarter


In [0]:
students_with_scholarships = df_stu_ljoin.join(df_schol_ljoin, on="student_id", how="left")
display(students_with_scholarships)

student_id,name,grade,scholarship_type
1,Juan,85,Full
2,Ana,90,Half
3,Luis,78,null
4,Marta,92,Full
5,Carlos,87,Half
6,Elena,88,null
7,Pedro,75,Quarter
8,Lucía,91,null
9,Miguel,80,Half
10,Sara,89,null


## Ejercicio 7:

Combina la lista de inscripciones con la lista de cursos usando un `RIGHT JOIN`. Esto permite conservar **todos los cursos**, incluso los que **no tienen estudiantes inscritos**.

### ✅ Instrucciones

1️⃣ Carga ambos archivos como DataFrames en PySpark.

2️⃣ Realiza un `RIGHT JOIN` de [`enrollments`](https://tajamar365.sharepoint.com/:x:/s/3431-MasterIA2025-2026/Edf7DeVH13VHrUwAYwyCPSYBEWP9fKKelLj3GALy0Z6u6Q?e=4BPXmd) con [`courses`](https://tajamar365.sharepoint.com/:x:/s/3431-MasterIA2025-2026/EfwXRSWHSS5Hu0f4Zu3rQ54B3EDyMRRrPk1bV0NZPaKQQg?e=1tmchi) usando la columna `course_id`.

3️⃣ Guarda el resultado como `enrollments_with_courses`.

4️⃣ Muestra el resultado y verifica que **los cursos sin estudiantes aparecen con valores `null` en `student_name`**.

</aside>

In [0]:
df_enrollments = spark.read.csv("/Workspace/Users/inigosamuel.jimenez@tajamar365.com/enrollments.csv", header=True, inferSchema=True)
display(df_enrollments)
df_courses = spark.read.csv("/Workspace/Users/inigosamuel.jimenez@tajamar365.com/courses.csv", header=True, inferSchema=True)
display(df_courses)

student_name,course_id
Juan,1
Ana,1
Luis,2
Marta,2
Carlos,3
Elena,3
Pedro,4
Lucía,5
Miguel,5
Sara,6


course_id,course_name
1,Math
2,Physics
3,Chemistry
4,Biology
5,History
6,Geography
7,Philosophy
8,Art
9,Music
10,PE


In [0]:
enrollments_with_courses = df_enrollments.join(df_courses, on="course_id", how="right")
display(enrollments_with_courses)

course_id,student_name,course_name
1,Ana,Math
2,Marta,Physics
3,Elena,Chemistry
4,Pedro,Biology
5,Miguel,History
6,Sara,Geography
7,null,Philosophy
8,null,Art
9,null,Music
10,null,PE


## Ejercicio 8:

Combina la lista de empleados asignados con la lista de proyectos usando un `OUTER JOIN`. Esto permite conservar **todos los proyectos** y **todos los empleados**, incluso si **no están relacionados**.

### ✅ Instrucciones

1️⃣ Carga ambos archivos como DataFrames en PySpark.

2️⃣ Realiza un `OUTER JOIN` de [`assignments`](https://tajamar365.sharepoint.com/:x:/s/3431-MasterIA2025-2026/EeXdii_WuHNKtPdUJQ7P268B5jeA1ySQAmCCVNA9QYGk1A?e=G0gkoF) con [`projects`](https://tajamar365.sharepoint.com/:x:/s/3431-MasterIA2025-2026/EaDlyza_NrJBoP5glLXxLHgBCXH7LQgnrZmqMH8s8sfj4w?e=cqAbmY) usando la columna `project_id`.

3️⃣ Guarda el resultado como `assignments_with_projects`.

4️⃣ Muestra el resultado y verifica que aparezcan:

- Proyectos sin empleados asignados.
- Empleados asignados a proyectos inexistentes.
- Empleados sin proyecto (valores `null`).
</aside>

In [0]:
df_assignments = spark.read.csv("/Workspace/Users/inigosamuel.jimenez@tajamar365.com/assignments.csv", header=True, inferSchema=True)
display(df_assignments)
df_projects = spark.read.csv("/Workspace/Users/inigosamuel.jimenez@tajamar365.com/projects.csv", header=True, inferSchema=True)
display(df_projects)

employee_name,project_id
Juan,1
Ana,1
Luis,2
Marta,2
Carlos,3
Elena,3
Pedro,16
Lucía,17
Miguel,2
Sara,null


project_id,project_name
1,Alpha
2,Beta
3,Gamma
4,Delta
5,Epsilon
6,Zeta
7,Eta
8,Theta
9,Iota
10,Kappa


In [0]:
assignments_with_projects = df_assignments.join(df_projects, on="project_id", how="outer")
display(assignments_with_projects)

project_id,employee_name,project_name
14,null,Xi
1,Ana,Alpha
6,Diego,Zeta
15,null,Omicron
4,David,Delta
5,Paula,Epsilon
8,null,Theta
10,null,Kappa
7,Nuria,Eta
13,null,Nu
